In [1]:
import torch

In [2]:
# Dans PyTorch, le tenseur est la brique de base universelle (building block). Tout est uniformisé sous forme de tenseur, qu'il s'agisse :
#  - d'un simple nombre (scalaire)
#  - d'une liste de nombres (vecteur)
#  - d'une image (matrice 2D)
#  - d'une vidéo (tenseur 4D)
# simples nombres emballés dans un tenseur, 
# ce qui leur permet de profiter de la machinerie de PyTorch (graphe de calcul, compatibilité GPU, etc.) tout en restant un scalaire.
x, y = torch.tensor(2.), torch.tensor(7.)

In [5]:
# requires_grad (booléen) indique à PyTorch s'il doit suivre les opérations faites sur ce tenseur 
#  pour calculer automatiquement les gradients plus tard.# 
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [7]:
# Loss function : Elle quantifie l'erreur de votre modèle.
#  Elle mesure à quel point la prédiction du modèle s'éloigne de la vraie valeur cible (y).
#  Squared Error
L = (w * x + b - y) ** 2

L.backward() lance la rétropropagation du gradient (backpropagation).       
- Elle calcule les dérivées (gradients) : En partant de la perte L, elle parcourt le graphe de calcul à l'envers 
  (grâce à la règle de dérivation en chaîne / chain rule) pour mesurer l'impact de chaque paramètre (w et b) sur l'erreur commise.
- Elle stocke les résultats : Elle enregistre ces gradients directement dans l'attribut .grad des tenseurs concernés 
  (c'est-à-dire ceux qui ont l'attribut requires_grad=True).

PyTorch calcule deux dérivées : comment $L$ varie si on bouge un tout petit peu $w$, et comment $L$ varie si on bouge un tout petit peu $b$.

Pour le poids $w$ ($\frac{\partial L}{\partial w}$) :$$\frac{\partial L}{\partial w} = 2 \cdot (w \cdot x + b - y) \cdot x$$

Pour le biais $b$ ($\frac{\partial L}{\partial b}$) :$$\frac{\partial L}{\partial b} = 2 \cdot (w \cdot x + b - y) \cdot 1$$

In [ ]:
# Une fois ces calculs de dérivées terminés, PyTorch va ranger ces résultats directement là où il faut :
# - il stocke -20. dans l'attribut w.grad.
# - il stocke -10. dans l'attribut b.grad.
L.backward()
print(w.grad, b.grad)

tensor(-20.) tensor(-10.)


##### Une fois les gradients calcules, PyTorch détruit le graphe de calcul de l'exemple L = (w * x + b - y) et libère toute la mémoire associée. 
- Si on décompose cela sous forme de graphe, PyTorch ne fait pas tout en un seul bloc. Il découpe le calcul en petites étapes successives :
    - Étape 1 (Multiplication) : On multiplie w par x $\rightarrow$ cela donne un résultat intermédiaire (disons $A = w \times x$).
    - Étape 2 (Addition) : On ajoute b à ce résultat $A$ $\rightarrow$ cela donne la prédiction ($B = A + b$, ou $\hat{y}$).
    - Étape 3 (Soustraction) : On soustrait y à cette prédiction $\rightarrow$ cela donne l'écart ($C = B - y$).
    - Étape 4 (Puissance) : On élève cet écart $C$ au carré $\rightarrow$ cela donne la perte finale (L).

- Forward pass : pendant le trajet aller, PyTorch avance de gauche à droite (de w, x, b, y vers L) en enregistrant chaque maillon de la chaîne dans sa mémoire. Il construit le graphe.
- Retropropragation : pendant le trajet retour, PyTorch repart de la fin (L) et remonte le graphe à l'envers, de droite à gauche, en appliquant la règle de dérivation en chaîne (chain rule) sur chaque petit maillon pour propager l'erreur jusqu'à w et b.

In [ ]:
# Si les gradients existent, on les remet à zéro
if w.grad is not None:
    w.grad.zero_()
if b.grad is not None:
    b.grad.zero_()

##### retain_graph=True permet de garder le graphe intact en arrière-plan au cas où on aurait encore besoin besoin de lui.

In [22]:
L = (w * x + b - y) ** 2

retain_graph=False
L.backward(retain_graph=retain_graph)
print(w.grad, b.grad)

tensor(-20.) tensor(-10.)


##### Forward pass :
- L'information avance de l'entrée vers la sortie :
    - Vous prenez vos données d'entrée ($x$).
    - La première couche applique ses poids et son biais : $z_1 = w_1 \cdot x + b_1$.
    - Le résultat passe par une fonction d'activation, puis arrive à la couche suivante (ou couche de sortie).
    - À la toute fin, on compare la sortie finale avec la vraie valeur ($y$) grâce à la fonction de perte ($L$).
    
##### Backward Pass : 
- C'est là que le gradient intervient, en sens inverse (de la sortie vers l'entrée) :
    - On part de la perte finale $L$.
    - On calcule le gradient de la dernière couche.
    - Grâce à la règle de dérivation en chaîne (chain rule), ce gradient est rétro-propagé à la couche précédente, puis à celle d'avant, et ainsi de suite.

### La rétropropagation calcule combien il faut changer les poids, et la descente de gradient effectue réellement le changement.

Voici les 6 principaux optimiseurs que l'on retrouve dans le module torch.optim de PyTorch :

1. torch.optim.SGD (Stochastic Gradient Descent)

    - Ce que c'est : La descente de gradient stochastique classique, avec ou sans momentum (inertie).   Quand l'utiliser : Très populaire en vision par ordinateur (CNN) et sur des problèmes bien balisés, souvent combiné avec du momentum (momentum=0.9).

2. torch.optim.Adam (Adaptive Moment Estimation)

    - Ce que c'est : L'un des rois incontestés du Deep Learning. Il adapte automatiquement le taux d'apprentissage (learning rate) pour chaque paramètre individuellement en fonction de l'historique des gradients.   Quand l'utiliser : Le choix par défaut idéal pour la grande majorité des architectures (NLP, réseaux denses, prototypage rapide) car il converge vite et demande peu de réglages fins.   

3. torch.optim.AdamW (Adam with Decoupled Weight Decay)

    - Ce que c'est : Une variante améliorée d'Adam. Elle sépare proprement la régularisation (le weight decay, qui évite le surentraînement) de la mise à jour des gradients.   Quand l'utiliser : C'est aujourd'hui le standard absolu pour entraîner les grands modèles de langage (LLM), les Transformers et la majorité des réseaux modernes.   

4. torch.optim.RMSprop (Root Mean Square Propagation)

    - Ce que c'est : Un optimiseur adaptatif qui divise le taux d'apprentissage par une moyenne mobile des magnitudes récentes du gradient.   Quand l'utiliser : Très performant pour les réseaux récurrents (RNN, LSTM) et les fonctions de coût "non stationnaires" (notamment en Reinforcement Learning / Apprentissage par renforcement).   

5. torch.optim.Adagrad (Adaptive Gradient Algorithm)

    - Ce que c'est : Il adapte le taux d'apprentissage en le réduisant pour les paramètres qui reçoivent de fréquentes mises à jour, et en l'augmentant pour les rares.   Quand l'utiliser : Idéal pour traiter des données très éparses (sparse data), par exemple en traitement du langage naturel (NLP) avec des matrices de mots creuses ou des systèmes de recommandation.   

6. torch.optim.LBFGS (Limited-memory BFGS)

    - Ce que c'est : Un optimiseur de second ordre. Au lieu de regarder uniquement la pente (1ère dérivée), il approche la courbure (2e dérivée / matrice hessienne) pour faire des pas beaucoup plus précis.   Quand l'utiliser : Réservé aux petits modèles ou aux tâches d'optimisation déterministes (comme le style transfer ou l'ajustement de courbes), car il est extrêmement gourmand en mémoire et nécessite souvent de recalculer la fonction entière (closure).   

In [ ]:
import torch

# 1. Données d'entrée et cible (ground truth)
x = torch.tensor(2.0)
y = torch.tensor(7.0)

# 2. Paramètres initiaux
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 3. Optimiseur (Descente de gradient)
learning_rate = 0.05
optimizer = torch.optim.SGD([w, b], lr=learning_rate)

# 4. LA BOUCLE D'ENTRAÎNEMENT (par exemple, 10 tours)
epochs = 10

for epoch in range(epochs):
    # Remettre les gradients à zéro
    optimizer.zero_grad()
    
    # Forward pass (Prédiction et calcul de la Perte / Loss)
    prediction = w * x + b
    loss = (prediction - y) ** 2
    
    # Rétropropagation (Calcul des gradients)
    #  PyTorch parcourt le graphe à l'envers et remplit w.grad et b.grad
    # loss.backward() ne remplit pas seulement les .grad d'une seule couche. 
    #  Il remplit les .grad de TOUTES les couches du réseau en une seule fois, de la dernière jusqu'à la première.
    loss.backward()
    
    # Descente de gradient (Mise à jour des poids)
    #  L'optimiseur utilise w.grad et b.grad pour modifier w et b selon le learning_rate
    optimizer.step()
    
    # Affichage pour suivre l'évolution
    print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

Epoch  1 | Loss: 25.0000 | w: 2.0000 | b: 0.5000
Epoch  2 | Loss: 6.2500 | w: 2.5000 | b: 0.7500
Epoch  3 | Loss: 1.5625 | w: 2.7500 | b: 0.8750
Epoch  4 | Loss: 0.3906 | w: 2.8750 | b: 0.9375
Epoch  5 | Loss: 0.0977 | w: 2.9375 | b: 0.9688
Epoch  6 | Loss: 0.0244 | w: 2.9688 | b: 0.9844
Epoch  7 | Loss: 0.0061 | w: 2.9844 | b: 0.9922
Epoch  8 | Loss: 0.0015 | w: 2.9922 | b: 0.9961
Epoch  9 | Loss: 0.0004 | w: 2.9961 | b: 0.9980
Epoch 10 | Loss: 0.0001 | w: 2.9980 | b: 0.9990


##### Le moteur d'autodifférenciation de PyTorch traite le calcul des gradients comme un problème de théorie des graphes.

1. Le Graphe de Calcul en tant que DAG (Directed Acyclic Graph) : Durant la phase aller (Forward pass), PyTorch ne se contente pas de stocker des nombres : il construit dynamiquement un DAG (Graphe Orienté Acyclique) :

    - Les Noeuds (Nodes) : Représentent les opérations mathématiques exécutées (multiplications, additions, fonctions d'activation, etc.). Chaque objet tenseur possède un attribut pointer-vers-son-origine nommé grad_fn (ou pointe vers un objet Node en C++).

    - Les Arêtes (Edges) : Représentent les flux de données (les tenseurs intermédiaires) qui transitent d'une opération à l'autre.

2. Le Tri Topologique Inverse (Reverse Topological Sort) : Lorsque vous appelez loss.backward(), le moteur C++ planifie et exécute un tri topologique inverse sur ce DAG :

    - Pourquoi un tri topologique ? Un réseau de neurones moderne n'est pas une simple séquence linéaire. Avec des architectures complexes (comme les ResNets et leurs skip connections, ou les U-Net), le graphe comporte des bifurcations, des fusions et des boucles conditionnelles.

    - Le tri topologique garantit un ordre d'exécution strict : le moteur ne peut rétro-propager l'erreur à travers un noeud parent qu'à la condition stricte que tous les noeuds enfants (les dépendances aval) ont déjà calculé et accumulé leurs gradients.

3. La Gestion des Flux Multiples (Forking et Merging) : Dans une architecture non linéaire (par exemple, une connexion résiduelle où un tenseur est à la fois injecté dans une couche profonde et ajouté plus loin via une addition directe) :

    - La divergence (Forking) : Lors du backward(), un gradient issu de la perte peut se diviser en plusieurs branches. Le moteur sait dupliquer et propager le signal le long de chaque chemin ascendant.

    - La convergence (Merging) : Inversement, si plusieurs chemins fusionnent (comme dans un point d'addition de ResNet), les gradients provenant de ces différentes branches doivent être additionnés au même endroit. Le moteur gère nativement cette accumulation de gradients dans les tenseurs partagés.

4. L'Exécution Locale : Les Produits Vecteur-Jacobien (VJP) : À chaque noeud du graphe traversé en sens inverse, le moteur n'applique pas une formule magique globale, mais exécute localement le Produit Vecteur-Jacobien (VJP - Vector-Jacobian Product) dicté par la règle de dérivation en chaîne (chain rule).
Chaque type d'opération possède sa propre méthode de rétro-propagation implémentée en C++ (par exemple, la fonction de rétropropagation d'une matrice de convolution, d'un ReLU, etc.), ce qui permet au graphe de traverser n'importe quel type de couche de manière modulaire.

##### Avant l'entraînement : L'initialisation aléatoireAu tout début (avant même de lancer le premier calcul), les poids $W$ et les biais $b$ de chaque couche ne valent absolument rien de intelligent. Ils sont initialisés de manière aléatoire (petits nombres tirés d'une loi normale ou uniforme via des méthodes comme Xavier ou Kaiming).

1. Pendant le Forward Pass : $W$ et $b$ sont fixes

   - Pendant que le signal traverse le réseau à l'aller :
   
        - Les $W$ et $b$ ne sont pas calculés ni modifiés.
        - Ils sont utilisés comme des constantes figées pour cette étape. Le réseau se contente de faire bêtement son calcul : $z = W \cdot x + b$ avec les valeurs qu'il a en main à ce moment précis.

2. La correction : Le duo backward() + optimizer

   - C'est là que tout s'articule :
   
      - La Loss mesure à quel point la prédiction (faite avec ces poids aléatoires) est mauvaise.
      - Le Backward calcule les gradients ($\frac{\partial L}{\partial W}$ et $\frac{\partial L}{\partial b}$) et les stocke dans W.grad et b.grad. Ces gradients indiquent la direction et la force de l'erreur pour chaque poids.
      - L'optimiseur (optimizer.step()) arrive alors en fin de course, prend ces gradients, et met à jour les poids :$$\text{Nouveau } W = \text{Ancien } W - \text{learning\_rate} \times W.grad$$

##### Ce qui se passe pour une couche donnée lors du passage de la rétropropagation (backward), en distinguant ce qui est stocké en mémoire (les gradients des paramètres) et ce qui est transmis en arrière (le signal pour la couche précédente).

1. Ce qui est stocké dans la couche elle-même (pour l'optimiseur)

    - Le gradient par rapport aux poids ($W$) : Stocké dans W.grad
    
        $$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \cdot x^T$$

        Note : Pendant le Forward pass, le signal traverse le réseau de gauche à droite. À chaque fois qu'une couche fait son calcul, elle met son propre $x$ (son entrée à elle) dans sa propre boîte de sauvegarde (ctx) :
        
        - Couche 1 sauvegarde son entrée ($x_1$, qui est l'input brut de 6 features).
        - Couche 2 sauvegarde son entrée ($x_2$, qui est la sortie de la couche 1).
        - Couche 3 sauvegarde son entrée ($x_3$, qui est la sortie de la couche 2).Et ainsi de suite jusqu'à la dernière couche.

    - Le gradient par rapport au biais ($b$) : Stocké dans b.grad
    
        $$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z}$$

2. Ce qui est transmis vers la couche précédente : pour que la couche d'avant sache à son tour quel a été son impact sur l'erreur, la couche actuelle transforme le signal d'erreur et le renvoie vers l'amont :

    - Le signal de base (linéaire) transmis à l'entrée $x$ :
    
        $$\frac{\partial L}{\partial x} = W^T \cdot \frac{\partial L}{\partial z}$$

    - Le signal complet (si la couche inclut une fonction d'activation comme ReLU) : Si le signal doit traverser une fonction d'activation en sens inverse, on le filtre avec la dérivée de cette activation (via le produit élément par élément $\odot$) :
    
        $$\frac{\partial L}{\partial z_{prev}} = \left( W^T \cdot \frac{\partial L}{\partial z} \right) \odot \text{ReLU}'(z_{prev})$$

